# AELIONIX BLACKFORGE — Phase 11 Colab Validation

This notebook performs a deterministic, one-click validation of the **Cloud
Security Capability Foundation** (Phase 11).

It exercises the full `blackforge.cloud` pipeline on three mock estates —
**AWS** (`aws/aelionix-aws-test`), **Azure** (`azure/aelionix-azure-test`) and
**GCP** (`gcp/aelionix-gcp-test`):

* **provider_discovery / account_inventory / project_inventory /
  resource_inventory** — provider identity, accounts, projects, and the
  resource catalog
* **compute / storage / database / network / container / cluster observation**
  — typed resource observation with `contains` / `located_in` structure
* **public_exposure_analysis / security_configuration_observation /
  secret_reference_observation** — exposure flags, security configuration
  (contradictions surface instead of silently overwriting), and secret
  references (credential values redacted before any row persists)
* **iam_identity / iam_role / iam_permission observation** — directory of
  identities, roles, and permissions
* **resource_relationship_analysis** — structural edges only; no attack-graph
  vocabulary is ever materialized
* **edge_architecture_observation / origin_candidate_analysis /
  transport_security_observation** — the Phase 11 amendment: which edge
  fronts which origin (`PROTECTS` / `PROXIES` / `FRONTED_BY`), origin-candidate
  correlation that **never confirms** (a HIGH-confidence candidate stays at
  `INFERRED` / `unvalidated` until validated), and contradictory TLS rows
  (`tls_enforced = True` vs `False`) that both surface.

Every capability runs the same guarded pipeline: request validation, scope /
authorization, resolution, **credential-redacted mock transport** (no real
cloud is ever queried or mutated), normalization, evidence persistence,
world-model materialization, and best-effort memory linking.

> Run all cells top-to-bottom. No GPU, no external services, no credentials.
> The notebook fails loudly on any check.

---

In [ ]:
import sys
import platform

print("Blackforge Phase 11 Colab Validation (Cloud Security Capability Foundation)")
print("=" * 60)
print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("Architecture:", platform.machine())
print("=" * 60)

assert sys.version_info >= (3, 10), f"Blackforge requires Python 3.10+, got {sys.version}"
print("Python version check: PASS")

---

In [ ]:
from pathlib import Path
import subprocess
import sys
import os
import shutil

# -- Configuration (edit here if fork changes) ---------------------------
REPO_URL = "https://github.com/Sagelord00000001/Blackforge.git"
REPO_DIR = Path("/content/blackforge")
# -----------------------------------------------------------------------

if REPO_DIR.exists() and (REPO_DIR / "blackforge" / "__init__.py").exists():
    print(f"Repository already exists at {REPO_DIR}, updating...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
        check=True,
    )

os.chdir(str(REPO_DIR))
print(f"Repository ready at {REPO_DIR}")

---

In [ ]:
import subprocess

try:
    commit = subprocess.run(
        ["git", "rev-parse", "--short", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print("Commit:", commit)
except Exception as e:
    print("Commit unavailable (expected in scratch checkouts):", e)

---

In [ ]:
!pip install hatchling --quiet
!pip install -e ".[dev]" --quiet

---

In [ ]:
import importlib

modules = [
    "blackforge",
    "blackforge.core.config",
    "blackforge.core.errors",
    "blackforge.core.types",
    "blackforge.runtime.bootstrap",
    "blackforge.capabilities.registry",
    "blackforge.authorization",
    "blackforge.scope.models",
    "blackforge.evidence",
    "blackforge.evidence.models",
    "blackforge.evidence.store",
    "blackforge.evidence.repository",
    "blackforge.world_model",
    "blackforge.world_model.models",
    "blackforge.world_model.query",
    "blackforge.world_model.repository",
    "blackforge.world_model.store",
    "blackforge.cloud",
    "blackforge.cloud.models",
    "blackforge.cloud.capabilities",
    "blackforge.cloud.transport",
    "blackforge.cloud.redaction",
    "blackforge.cloud.evidence",
    "blackforge.cloud.normalization",
    "blackforge.cloud.materializer",
    "blackforge.cloud.engine",
    "blackforge.cloud.addressing",
    "blackforge.cloud.providers",
]

_import_failures = []
for module in modules:
    try:
        importlib.import_module(module)
    except Exception as e:
        _import_failures.append((module, str(e)))

if _import_failures:
    for mod, err in _import_failures:
        print(f"  FAIL: {mod} — {err}")
    raise RuntimeError(f"Import health check failed: {len(_import_failures)} module(s)")

print(f"Blackforge imports OK ({len(modules)} modules verified).")
print("Cloud module imports: PASS")

---

In [ ]:
import subprocess
import sys

print("Running automated test suite...")
# The LLM/torch-heavy files are excluded: importing the HF provider pulls
# ~2GB of torch memory and can SIGKILL the kernel on CPU runtimes. Those
# tests are validated locally and in the Phase 1 notebook.
result = subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q", "--tb=short",
        "--ignore=tests/test_huggingface_provider.py",
        "--ignore=tests/test_loader.py",
        "--ignore=tests/test_smoke_real_model.py",
    ],
    capture_output=True, text=True, cwd=str(REPO_DIR),
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")
    raise RuntimeError(f"pytest failed with exit code {result.returncode}")

print("Automated test suite: PASS")

---

In [ ]:
import os
from pathlib import Path

DBROOT = Path("data/phase11_colab").resolve()
DBROOT.mkdir(parents=True, exist_ok=True)
os.environ["BLACKFORGE_DB_PATH"] = str(DBROOT / "blackforge.db")
os.environ["BLACKFORGE_MEMORY_DB_PATH"] = str(DBROOT / "memory.db")
os.environ["BLACKFORGE_EVIDENCE_DB_PATH"] = str(DBROOT / "evidence.db")
os.environ["BLACKFORGE_WORLD_MODEL_DB_PATH"] = str(DBROOT / "world_model.db")
for _p in (DBROOT / "evidence.db", DBROOT / "world_model.db"):
    _p.unlink(missing_ok=True)

from blackforge.runtime.bootstrap import bootstrap

app = bootstrap()
assert app.healthy(), "Blackforge health check failed"
verification = app.verify()
for key in ("config_loaded", "mission_manager_ready", "capability_registry_ready",
            "memory_ready", "evidence_store_ready", "evidence_memory_link_ready",
            "world_model_ready", "recon_ready", "webapi_ready", "auth_ready",
            "business_logic_ready", "network_ready", "identity_ready",
            "authorization_ready", "model_router_ready", "cloud_ready"):
    assert verification[key], f"{key} must be True"
assert verification["cloud_ready"] is True, "cloud_ready must be True (20 typed capabilities)"
assert len(app.capability_registry.list_capabilities()) == 81

BOOTSTRAP_OK = app.healthy() and bool(verification["cloud_ready"])

for k, v in verification.items():
    symbol = "PASS" if v else "FAIL"
    print(f"  [{symbol}] {k}")

print("\nBlackforge bootstrap (cloud_ready, 81 registered capabilities): PASS")

---

In [ ]:
from blackforge.cloud import CLOUD_CAPABILITY_IDS, CloudRequest, CloudMode
from blackforge.scope.models import TargetScope, Target
from blackforge.core.types import RiskLevel, TargetType

MID = "mission_phase11_cloud"

AWS = "aws/aelionix-aws-test"
AZURE = "azure/aelionix-azure-test"
GCP = "gcp/aelionix-gcp-test"
_ERROR_TARGETS = [
    "aws/snail-account",
    "aws/bursty-account",
    "aws/locked-account",
    "aws/garbled-account",
    "aws/fabricated-estate",
]
ALL = [AWS, AZURE, GCP, "aws", "oci/foo"] + _ERROR_TARGETS

scope = TargetScope(
    mission_id=MID,
    allowed_targets=[Target(value=t, target_type=TargetType.CLOUD) for t in ALL],
    max_risk_level=RiskLevel.HIGH,
)
req = CloudRequest(
    mission_id=MID, session_id="ses_phase11_cloud", scope=scope,
    mode=CloudMode.CONTROLLED, max_observations=500, timeout_seconds=30.0,
)

engine = app.cloud_engine
assert engine is not None and len(engine.capabilities) == 20
ids_seen = [c.capability_id for c in engine.capabilities]
assert set(ids_seen) == set(CLOUD_CAPABILITY_IDS), (ids_seen, CLOUD_CAPABILITY_IDS)
print("Registered cloud capabilities:", ", ".join(ids_seen))

for c in engine.capabilities:
    meta = c.meta()
    risk = meta.risk_level.value
    mode = meta.mode.value
    assert risk == "low", c.capability_id
    assert mode == "passive", c.capability_id
    assert meta.world_model, f"{c.capability_id} must materialize into the world model"
    assert TargetType.CLOUD in meta.supported_target_types, c.capability_id
    assert len(meta.produces) == 1, c.capability_id
    print(
        f"  {str(meta.id):<40} risk={risk:<7} mode={mode:<8} "
        f"targets={[t.value for t in meta.supported_target_types]}"
    )
CAPS_OK = True

---

In [ ]:
from blackforge.cloud import CloudStatus, observation_confidence
from blackforge.evidence.models import EvidenceRelation, EvidenceStatus, EvidenceType
from blackforge.core.types import Confidence
from blackforge.world_model.query import RelationshipQuery, WorldQuery
from blackforge.world_model.models import EntityType, RelationshipType, WorldLifecycle

TOOLS = [
    "discover_providers", "inventory_accounts", "inventory_projects", "inventory_resources",
    "observe_compute", "observe_storage", "observe_databases", "observe_networks",
    "analyze_public_exposure", "observe_security_configuration", "observe_secret_references",
    "observe_iam_identities", "observe_iam_roles", "observe_iam_permissions",
    "analyze_resource_relationships", "observe_containers", "observe_clusters",
    "observe_edge_architecture", "analyze_origin_candidates", "observe_transport_security",
]
EXPECTED_COUNTS: dict[str, dict[str, int]] = {
    # Engine-level observation counts. AWS observe_security_configuration is
    # 6 (PARTIAL: the cloudtrail contradiction pair merges into one status);
    # the raw transport document carries 7 rows before typed resolution.
    AWS: {
        "discover_providers": 1, "inventory_accounts": 1, "inventory_projects": 2,
        "inventory_resources": 17, "observe_compute": 2, "observe_storage": 3,
        "observe_databases": 1, "observe_networks": 5, "analyze_public_exposure": 5,
        "observe_security_configuration": 6, "observe_secret_references": 3,
        "observe_iam_identities": 3, "observe_iam_roles": 3, "observe_iam_permissions": 5,
        "analyze_resource_relationships": 7, "observe_containers": 2, "observe_clusters": 1,
        "observe_edge_architecture": 2, "analyze_origin_candidates": 4,
        "observe_transport_security": 4,
    },
    AZURE: {
        "discover_providers": 1, "inventory_accounts": 1, "inventory_projects": 2,
        "inventory_resources": 9, "observe_compute": 2, "observe_storage": 1,
        "observe_databases": 1, "observe_networks": 2, "analyze_public_exposure": 2,
        "observe_security_configuration": 2, "observe_secret_references": 1,
        "observe_iam_identities": 2, "observe_iam_roles": 2, "observe_iam_permissions": 2,
        "analyze_resource_relationships": 2, "observe_containers": 1, "observe_clusters": 1,
        "observe_edge_architecture": 1, "analyze_origin_candidates": 2,
        "observe_transport_security": 2,
    },
    GCP: {
        "discover_providers": 1, "inventory_accounts": 1, "inventory_projects": 1,
        "inventory_resources": 8, "observe_compute": 2, "observe_storage": 1,
        "observe_databases": 1, "observe_networks": 1, "analyze_public_exposure": 2,
        "observe_security_configuration": 2, "observe_secret_references": 1,
        "observe_iam_identities": 2, "observe_iam_roles": 2, "observe_iam_permissions": 2,
        "analyze_resource_relationships": 2, "observe_containers": 1, "observe_clusters": 1,
        "observe_edge_architecture": 1, "analyze_origin_candidates": 2,
        "observe_transport_security": 2,
    },
}


_default_status = {
    ("aws/aelionix-aws-test", "observe_security_configuration"): CloudStatus.PARTIAL,
}


def _pipeline_all(engine, req):
    runs = {}
    for target in (AWS, AZURE, GCP):
        for tool in TOOLS:
            result = getattr(engine, tool)(req, target)
            assert result.authorized is True, (target, tool)
            expected_status = _default_status.get((target, tool), CloudStatus.SUCCESS)
            assert result.status == expected_status, (target, tool, result.status)
            assert result.observation_count == EXPECTED_COUNTS[target][tool], (
                target, tool, result.observation_count,
            )
            runs[(target, tool)] = result
    return runs


runs = _pipeline_all(engine, req)
statuses = []
for target in (AWS, AZURE, GCP):
    for tool in TOOLS:
        r = runs[(target, tool)]
        statuses.append(f"{target.split('/')[0]}.{tool}={r.status.value}({r.observation_count})")
print("All 20 cloud capabilities executed on all 3 estates")
print("Statuses:", " ".join(statuses))
print(f"Total observations asserted: {sum(r.observation_count for r in runs.values())}")

# Every observation evidence row is DERIVED_FROM its run's artifact row.
rel_ok = True
count_obs = 0
for r in runs.values():
    artifact = r.evidence_ids[0]
    for ev_id in r.evidence_ids[1:]:
        rels = app.evidence_store.get_relationships(ev_id)
        ok = any(
            x.relation_type == EvidenceRelation.DERIVED_FROM
            and str(x.target_id) == str(artifact)
            for x in rels
        )
        rel_ok = rel_ok and ok
        count_obs += 1
assert rel_ok and count_obs >= 150, count_obs
print(f"DERIVED_FROM links: {count_obs} observations across {len(runs)} artifacts")

# Every row persists as OBSERVED (cloud evidence never elevates beyond observed).
stored = {e.id: e.status for e in app.evidence_store.list(limit=10000)}
assert all(stored[ev] == EvidenceStatus.OBSERVED for r in runs.values()
           for ev in r.evidence_ids), "cloud evidence must stay OBSERVED"
print("All cloud evidence rows persisted with status OBSERVED")

total_evidence = app.evidence_store.count(MID)
assert total_evidence > 200, total_evidence
print(f"Evidence rows for mission: {total_evidence}")

# --- world materialization --------------------------------------------------
wm = app.world_model
entities = wm.list_entities(WorldQuery(mission_id=MID, limit=1000))
etypes = {e.entity_type.value for e in entities}
needed = {
    "cloud_provider", "cloud_account", "cloud_project", "cloud_region",
    "cloud_compute", "cloud_storage", "cloud_database", "cloud_network",
    "cloud_secret", "cloud_container", "cloud_cluster",
    "identity", "role", "permission",
    "edge_endpoint", "origin_endpoint", "origin_candidate",
    "endpoint", "public_address", "private_address",
}
assert needed <= etypes, etypes
print("World entity types:", ", ".join(sorted(etypes)))

rels = wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000))
rel_types = {r.relationship_type.value for r in rels}
need_rel = {
    "contains", "located_in",
    "protects", "proxies", "fronted_by",
    "routes_to", "originates_from",
}
assert need_rel <= rel_types, rel_types
offensive = {
    "exploits", "can_compromise", "leads_to", "enables",
    "privilege_escalation_path",
}
assert rel_types & offensive == set(), rel_types & offensive
print("Relationship types:", ", ".join(sorted(rel_types)))
print("No attack-graph relationship types (EXPLOITS/CAN_COMPROMISE/LEADS_TO/ENABLES): PASS")

# Cross-estate headline totals (edge/origin/candidate/endpoint families).
_headline = {
    EntityType.EDGE_ENDPOINT: 4,
    EntityType.ORIGIN_ENDPOINT: 3,
    EntityType.ORIGIN_CANDIDATE: 8,
    EntityType.PUBLIC_ADDRESS: 3,
    EntityType.PRIVATE_ADDRESS: 4,
    EntityType.ENDPOINT: 7,
}
for entity_type, expected_count in _headline.items():
    actual = wm.count_entities(MID, entity_type=entity_type, lifecycle=WorldLifecycle.ACTIVE)
    assert actual == expected_count, (entity_type, actual, expected_count)
    print(f"  {entity_type.value:<20} active={actual}")
assert wm.count_entities(MID, lifecycle=WorldLifecycle.ACTIVE) == 101

# Confidence policy: direct authoritative -> HIGH, derived -> MEDIUM, passive -> LOW.
compute_obs = runs[(AWS, "observe_compute")].observations[0]
assert observation_confidence(compute_obs, CloudMode.CONTROLLED) == Confidence.HIGH
edge_obs = runs[(AWS, "observe_edge_architecture")].observations[0]
assert observation_confidence(edge_obs, CloudMode.CONTROLLED) == Confidence.MEDIUM
assert observation_confidence(edge_obs, CloudMode.PASSIVE) == Confidence.LOW
print("Confidence policy (direct -> HIGH, edge/origin/transport derived -> MEDIUM, passive -> LOW): PASS")

# Security configuration contradiction surfaces instead of silently overwriting.
sec = runs[(AWS, "observe_security_configuration")]
assert sec.status == CloudStatus.PARTIAL, sec.status
assert sec.observation_count == 6, sec.observation_count
account = next(e for e in wm.list_entities(
    WorldQuery(mission_id=MID, entity_type=EntityType.CLOUD_ACCOUNT, limit=100))
    if e.name == "aelionix-aws-test")
acc_assertions = wm.list_assertions(str(account.id), lifecycle=WorldLifecycle.ACTIVE)
cloudtrail = [a for a in acc_assertions if a.property_key == "cloudtrail_logging"]
assert {a.property_value for a in cloudtrail} == {"enabled", "disabled"}
print("Security configuration contradiction surfaced (cloudtrail enabled+disabled): PASS")

---

In [ ]:
# Mission isolation: cloud work under a second mission is disjoint.
MID2 = "mission_phase11_cloud_other"
scope2 = TargetScope(
    mission_id=MID2,
    allowed_targets=[Target(value=AWS, target_type=TargetType.CLOUD)],
    max_risk_level=RiskLevel.HIGH,
)
req2 = CloudRequest(
    mission_id=MID2, session_id="ses_phase11_cloud_2", scope=scope2,
    mode=CloudMode.CONTROLLED, max_observations=500, timeout_seconds=30.0,
)
r2 = engine.observe_edge_architecture(req2, AWS)
other_ids = {str(x) for x in r2.evidence_ids}
assert other_ids.isdisjoint({str(x) for x in runs[(AWS, "observe_edge_architecture")].evidence_ids})
assert app.evidence_store.count(MID2) == len(r2.evidence_ids)
assert wm.count_entities(MID2, entity_type=EntityType.EDGE_ENDPOINT,
                         lifecycle=WorldLifecycle.ACTIVE) == 2
print("Mission isolation: second mission produced its own evidence/world rows: PASS")

# Redaction: the secret-references artifact preserves structure but strips secrets.
rows = {e.id: e for e in app.evidence_store.list(limit=10000)}
artifact = rows[runs[(AWS, "observe_secret_references")].evidence_ids[0]]
assert artifact.evidence_type == EvidenceType.ARTIFACT
assert "demo-" not in artifact.raw_data, artifact.raw_data
assert "REDACTED" in artifact.raw_data
print("Secret-reference artifact redacted (no credential values stored): PASS")

from blackforge.cloud import redact_cloud_raw
import json

demo_raw = json.dumps({
    "provider": "aws",
    "bucket": "orders-backup",
    "connection_string": "demo-connection-string-0000",
    "access_key_id": "demo-access-key-0000",
    "secret_value": "demo-secret-value-0000",
    "tags": ["backup"],
})
clean_raw = redact_cloud_raw(demo_raw)
clean = json.loads(clean_raw)
assert clean["connection_string"] == "REDACTED"
assert clean["access_key_id"] == "REDACTED"
assert clean["secret_value"] == "REDACTED"
assert clean["bucket"] == "orders-backup"
assert "demo-" not in clean_raw
print("Redaction unit behavior (credential-like fields -> stable REDACTED marker): PASS")

# Idempotency: a repeated full-pipeline run over the same targets reuses rows.
before = app.evidence_store.count(MID)
runs2 = _pipeline_all(engine, req)
after = app.evidence_store.count(MID)
assert after == before, (before, after)
print("Idempotent full-pipeline re-run (no duplicate evidence rows): PASS")

# Candidate correlation never confirms an origin.
candidate = wm.find_entity(
    MID, EntityType.ORIGIN_CANDIDATE, "app.aelionix.test:10.0.0.10", namespace=AWS)
assert candidate is not None
assert candidate.entity_type == EntityType.ORIGIN_CANDIDATE
by_key: dict[str, set[str]] = {}
for a in wm.list_assertions(str(candidate.id), lifecycle=WorldLifecycle.ACTIVE):
    by_key.setdefault(a.property_key, set()).add(a.property_value or "")
assert by_key["confidence_label"] == {"high"}, by_key
assert by_key["evidence_status"] == {"inferred"}, by_key
assert by_key["validation_status"] == {"unvalidated"}, by_key
origin = wm.find_entity(MID, EntityType.ORIGIN_ENDPOINT, "10.0.0.10", namespace=AWS)
assert origin is not None
out = wm.list_relationships(
    RelationshipQuery(mission_id=MID, source_entity_id=str(candidate.id), limit=100))
routes = [r for r in out if r.relationship_type == RelationshipType.ROUTES_TO]
assert len(routes) == 1
assert str(routes[0].target_entity_id) == str(
    wm.find_entity(MID, EntityType.PRIVATE_ADDRESS, "10.0.0.10", namespace=AWS).id)
correlated = [r for r in out if r.relationship_type == RelationshipType.ORIGINATES_FROM]
assert len(correlated) == 1
correlated_target = wm.get_entity(str(correlated[0].target_entity_id))
assert correlated_target is not None
assert correlated_target.entity_type == EntityType.ORIGIN_ENDPOINT
assert correlated_target.name == "10.0.0.10"
print("HIGH-confidence candidate app.aelionix.test:10.0.0.10 stays inferred/unvalidated, "
      "correlated to the origin endpoint but never confirmed: PASS")

# Exposure-feed candidate stays LOW / HYPOTHESIZED and never correlates to an origin.
feed_candidate = wm.find_entity(
    MID, EntityType.ORIGIN_CANDIDATE, "app.aelionix.test:203.0.113.10", namespace=AWS)
assert feed_candidate is not None
fbk: dict[str, set[str]] = {}
for a in wm.list_assertions(str(feed_candidate.id), lifecycle=WorldLifecycle.ACTIVE):
    fbk.setdefault(a.property_key, set()).add(a.property_value or "")
assert fbk["confidence_label"] == {"low"}, fbk
assert fbk["evidence_status"] == {"hypothesized"}, fbk
assert fbk["validation_status"] == {"unvalidated"}, fbk
from blackforge.cloud import AddressType, classify_address
assert classify_address("203.0.113.10") == AddressType.PUBLIC_ADDRESS
assert classify_address("10.0.0.10") == AddressType.PRIVATE_ADDRESS
public = wm.find_entity(MID, EntityType.PUBLIC_ADDRESS, "203.0.113.10", namespace=AWS)
assert public is not None
out = wm.list_relationships(
    RelationshipQuery(mission_id=MID, source_entity_id=str(feed_candidate.id), limit=100))
assert len([r for r in out
            if r.relationship_type == RelationshipType.ROUTES_TO
            and str(r.target_entity_id) == str(public.id)]) == 1
assert all(r.relationship_type != RelationshipType.ORIGINATES_FROM for r in out)
print("Exposure-feed candidate stays LOW/hypothesized with no ORIGINATES_FROM: PASS")

candidates = wm.list_entities(
    WorldQuery(mission_id=MID, entity_type=EntityType.ORIGIN_CANDIDATE,
               namespace=AWS, lifecycle=WorldLifecycle.ACTIVE, limit=100))
assert all(e.entity_type == EntityType.ORIGIN_CANDIDATE for e in candidates)
print(f"AWS origin candidates: {len(candidates)} (all typed ORIGIN_CANDIDATE, none confirmed): PASS")

# Edge architecture: edge fronts origin, protects the app, never exposes it directly.
edges = wm.list_entities(
    WorldQuery(mission_id=MID, entity_type=EntityType.EDGE_ENDPOINT,
               namespace=AWS, lifecycle=WorldLifecycle.ACTIVE, limit=100))
assert len(edges) == 2
web01 = wm.find_entity(MID, EntityType.CLOUD_COMPUTE, "web-01", namespace=AWS)
assert web01 is not None
fronted_any = False
for edge in edges:
    reachable = {}
    for a in wm.list_assertions(str(edge.id), lifecycle=WorldLifecycle.ACTIVE):
        if a.property_key == "directly_reachable_origin":
            reachable.setdefault(a.property_key, set()).add(a.property_value or "")
    assert reachable["directly_reachable_origin"] == {"false"}
    protects = wm.list_relationships(
        RelationshipQuery(mission_id=MID, source_entity_id=str(edge.id),
                          relationship_type=RelationshipType.PROTECTS, limit=100))
    assert len(protects) == 1
    protected_target = wm.get_entity(str(protects[0].target_entity_id))
    assert protected_target is not None
    assert protected_target.entity_type == EntityType.CLOUD_COMPUTE
    assert protected_target.name == "web-01"
    proxies = wm.list_relationships(
        RelationshipQuery(mission_id=MID, source_entity_id=str(edge.id),
                          relationship_type=RelationshipType.PROXIES, limit=100))
    assert len(proxies) == 1
    proxy_target = wm.get_entity(str(proxies[0].target_entity_id))
    assert proxy_target is not None
    assert proxy_target.entity_type == EntityType.ORIGIN_ENDPOINT
    assert proxy_target.name == "10.0.0.10"
    fronted = wm.list_relationships(
        RelationshipQuery(mission_id=MID, source_entity_id=str(proxy_target.id),
                          relationship_type=RelationshipType.FRONTED_BY, limit=100))
    for r in fronted:
        if str(r.target_entity_id) == str(edge.id):
            fronted_any = True
assert fronted_any, "edge must front the origin it proxies"
print("Both AWS edges protect web-01, proxy the origin endpoint, and front it: PASS")

# Transport security: contradictory TLS rows both surface.
endpoints = wm.list_entities(
    WorldQuery(mission_id=MID, entity_type=EntityType.ENDPOINT, limit=1000))
app_endpoint = next(e for e in endpoints if "app.aelionix.test" in e.name)
ebk: dict[str, set[str]] = {}
for a in wm.list_assertions(str(app_endpoint.id), lifecycle=WorldLifecycle.ACTIVE):
    ebk.setdefault(a.property_key, set()).add(a.property_value or "")
assert ebk["tls_enforced"] == {"False", "True"}, ebk
assert ebk["tls_version"] == {"TLS1.0", "TLS1.3"}, ebk
print("Contradictory TLS assertions surfaced (tls_enforced False+True, TLS1.0+TLS1.3): PASS")

---

In [ ]:
from blackforge.core.errors import AuthorizationError, CloudExecutionError

# 1) Target outside the scope is denied BEFORE any transport runs.
denied_out = True
try:
    engine.observe_compute(req, "oci/oracle-account")
    denied_out = False
except AuthorizationError:
    pass
assert denied_out, "out-of-scope target oci/oracle-account must be denied"
print("Out-of-scope target denied before transport execution: PASS")

# 2) Non-cloud target type is rejected (no generic execution surface).
unsupported_type = True
try:
    engine.inventory_accounts(req, "api-service.example.com")
    unsupported_type = False
except CloudExecutionError:
    pass
assert unsupported_type, "non-cloud target type must be rejected"
print("Non-cloud target type rejected (no generic execution surface): PASS")

# 3) Unknown capability is rejected.
unknown_rejected = True
try:
    engine.run(req, "cloud.does_not_exist", AWS)
    unknown_rejected = False
except CloudExecutionError:
    pass
assert unknown_rejected, "unknown capability must be rejected"
print("Unknown capability rejected (no generic execution surface): PASS")

# 4) Invalid mode is rejected.
invalid_mode = True
try:
    engine.observe_compute(req, AWS, mode="turbo")
    invalid_mode = False
except CloudExecutionError:
    pass
assert invalid_mode, "invalid mode must be rejected"
print("Invalid mode rejected: PASS")

# 5) Failure states on the mock synthetic error estates.
from blackforge.cloud import CloudStatus

_error_map = {
    "aws/snail-account": CloudStatus.TIMEOUT,
    "aws/bursty-account": CloudStatus.RATE_LIMITED,
    "aws/locked-account": CloudStatus.UNAUTHORIZED,
    "aws/garbled-account": CloudStatus.MALFORMED_RESPONSE,
    "aws/fabricated-estate": CloudStatus.UNSUPPORTED_PROVIDER,
}
for target, expected in _error_map.items():
    got = engine.inventory_accounts(req, target)
    assert got.status == expected, (target, got.status, expected)
print("Failure state mapping (5 synthetic error estates) verified: PASS")

# 6) Unknown provider in scope fails closed with no observations.
scope_oci = TargetScope(
    mission_id=MID,
    allowed_targets=[Target(value="oci/foo", target_type=TargetType.CLOUD)],
    max_risk_level=RiskLevel.HIGH,
)
req_oci = CloudRequest(
    mission_id=MID, session_id="ses_phase11_cloud_oci", scope=scope_oci,
    mode=CloudMode.CONTROLLED, max_observations=500, timeout_seconds=30.0,
)
unknown_provider = engine.inventory_accounts(req_oci, "oci/foo")
assert unknown_provider.status == CloudStatus.UNKNOWN_PROVIDER
assert len(unknown_provider.observations) == 0
print("Unknown cloud provider fails closed (UNKNOWN_PROVIDER, 0 observations): PASS")

# 7) Observation limit truncates instead of overflowing.
from blackforge.cloud import CloudRequest as _CR

req_lim = _CR(
    mission_id=MID, session_id="ses_phase11_cloud_lim", scope=scope,
    mode=CloudMode.CONTROLLED, max_observations=2, timeout_seconds=30.0,
)
limited = engine.inventory_resources(req_lim, AWS)
assert limited.status == CloudStatus.LIMITED
assert limited.observation_count == 2
assert len(limited.warnings) == 1
print("Observation limit truncates (LIMITED, 2 observations, warning): PASS")

# 8) Passive mode is LOW confidence and never collides with controlled records.
req_pas = _CR(
    mission_id=MID, session_id="ses_phase11_cloud_pas", scope=scope,
    mode=CloudMode.PASSIVE, max_observations=500, timeout_seconds=30.0,
)
pas = engine.observe_compute(req_pas, AWS)
assert pas.mode == CloudMode.PASSIVE
assert observation_confidence(pas.observations[0], pas.mode) == Confidence.LOW
pas_ids = {str(x) for x in pas.evidence_ids}
assert pas_ids.isdisjoint({str(x) for x in runs[(AWS, "observe_compute")].evidence_ids})
print("Confidence mode policy (PASSIVE -> LOW, separate evidence rows): PASS")

---

In [ ]:
from blackforge.evidence.repository import SQLiteEvidenceRepository
from blackforge.evidence.store import EvidenceStore
from blackforge.world_model.repository import SQLiteWorldRepository
from blackforge.world_model.store import WorldModelStore

# Fresh connections over the same SQLite files prove restart persistence.
fresh_ev = EvidenceStore(SQLiteEvidenceRepository(str(DBROOT / "evidence.db")))
fresh_wm = WorldModelStore(SQLiteWorldRepository(str(DBROOT / "world_model.db")))

persisted_ev = fresh_ev.count(MID) == app.evidence_store.count(MID)
persisted_wm = fresh_wm.count_entities(MID) == wm.count_entities(MID)
candidate = fresh_wm.find_entity(
    MID, EntityType.ORIGIN_CANDIDATE, "app.aelionix.test:10.0.0.10", namespace=AWS)
persisted_candidate = candidate is not None
rel_count = len(fresh_wm.list_relationships(RelationshipQuery(mission_id=MID, limit=1000)))
assert persisted_ev and persisted_wm and persisted_candidate and rel_count > 0
assert fresh_ev.count(MID) > 0
PERSIST_OK = persisted_ev and persisted_wm and persisted_candidate

for store in (fresh_ev, fresh_wm):
    store.close()
try:
    app.evidence_store.close()
except Exception:
    pass
try:
    app.world_model.close()
except Exception:
    pass
print("Restart persistence (fresh connections on same DB files): PASS")
print("Backends closed. Validation summary below.")

---

In [ ]:
results = {}
phase_checks = {
    "repository_integrity": (REPO_DIR / "blackforge" / "cloud" / "engine.py").exists(),
    "phase11_modules": bool(
        (REPO_DIR / "blackforge" / "cloud" / "capabilities.py").exists()
        and (REPO_DIR / "blackforge" / "cloud" / "evidence.py").exists()
        and (REPO_DIR / "blackforge" / "cloud" / "materializer.py").exists()
        and (REPO_DIR / "blackforge" / "cloud" / "redaction.py").exists()
        and (REPO_DIR / "blackforge" / "cloud" / "normalization.py").exists()
        and (REPO_DIR / "blackforge" / "cloud" / "transport.py").exists()
        and (REPO_DIR / "blackforge" / "cloud" / "addressing.py").exists()
        and (REPO_DIR / "blackforge" / "cloud" / "providers.py").exists()
    ),
    "imports": len(_import_failures) == 0,
    "bootstrap_cloud_ready": BOOTSTRAP_OK,
    "capability_surface": CAPS_OK,
    "pipeline_evidence": rel_ok,
    "evidence_observed": bool(count_obs >= 150),
    "world_materialized": bool(needed <= etypes),
    "no_attack_graph": not bool(rel_types & offensive),
    "confidence_policy": True,
    "scope_authorization": denied_out,
    "unsupported_type_rejected": unsupported_type,
    "unknown_capability_rejected": unknown_rejected,
    "invalid_mode_rejected": invalid_mode,
    "redaction_boundary": True,
    "candidate_not_confirmed": bool(by_key["validation_status"] == {"unvalidated"}),
    "feed_candidate_hypothesized": bool(fbk["evidence_status"] == {"hypothesized"}),
    "mission_isolation": bool(other_ids.isdisjoint({str(x) for x in runs[(AWS, "observe_edge_architecture")].evidence_ids})),
    "idempotent_runs": bool(after == before),
    "restart_persistence": PERSIST_OK,
}

# The pytest cell aborts the run on failure, so reaching this cell proves it passed.
pytest_passed = True
install_ok = len(_import_failures) == 0

results["Repository"] = phase_checks["repository_integrity"]
results["Python"] = sys.version_info >= (3, 10)
results["Hardware"] = True  # CPU fallback always works; this notebook needs no GPU
results["Installation"] = install_ok
results["Imports"] = install_ok
results["Automated tests"] = pytest_passed
results["Bootstrap"] = phase_checks["bootstrap_cloud_ready"]
results["Phase-specific tests"] = all(phase_checks.values())
results["Security checks"] = (
    phase_checks["scope_authorization"]
    and phase_checks["unsupported_type_rejected"]
    and phase_checks["unknown_capability_rejected"]
    and phase_checks["invalid_mode_rejected"]
    and phase_checks["redaction_boundary"]
    and phase_checks["no_attack_graph"]
    and phase_checks["idempotent_runs"]
)

print()
print("=" * 60)
print("PHASE 11 COLAB VALIDATION SUMMARY")
print("=" * 60)
for name, ok in results.items():
    symbol = "PASS" if ok else "FAIL"
    print(f"  [{symbol}] {name}")

_all_ok = all(results.values()) and all(phase_checks.values())
assert _all_ok, "One or more validation checks failed"

print()
print("LOCAL VALIDATION: SUCCESS")
print()
print("Note: this notebook validates the commit checked out into /content/blackforge.")

---